# 第一章：环境搭建与核心概念

## 学习目标
- 使用 uv 搭建 LangChain 开发环境
- 配置通义千问（Qwen）API 并运行第一个程序
- 掌握消息类型（SystemMessage / HumanMessage / AIMessage）
- 理解 Runnable 统一接口（invoke / stream / batch）

## 1. 环境搭建（uv）

本项目使用 **uv** 管理 Python 环境和依赖。

```bash
# 在项目目录下初始化（首次）
uv init --python 3.11
uv add langchain langchain-core langchain-community langchain-openai

# 启动 Jupyter Lab
uv run --with jupyter jupyter lab
```

若已在 uv 环境中运行此 Notebook，直接执行下方单元格验证安装即可。

In [1]:
# 验证安装 & 版本确认
import langchain
import langchain_core
print(f"langchain 版本: {langchain.__version__}")
print(f"langchain-core 版本: {langchain_core.__version__}")

langchain 版本: 1.2.13
langchain-core 版本: 1.2.21


## 2. 配置 Qwen API Key

通义千问 API 与 OpenAI 接口兼容，Key 在 [DashScope 控制台](https://dashscope.console.aliyun.com/) 获取。

**推荐：写入 shell 配置文件（永久生效）**
```bash
echo 'export Qwen_API_KEY="sk-..."' >> ~/.bashrc && source ~/.bashrc
```

In [3]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

# 检查环境变量
if os.environ.get("Qwen_API_KEY"):
    print("✓ 已找到环境变量 Qwen_API_KEY")
else:
    print("✗ 未找到环境变量 Qwen_API_KEY，请先设置")

✓ 已找到环境变量 Qwen_API_KEY


## 3. 第一个 LangChain 程序

先看代码，再解释。

In [ ]:
from langchain_openai import ChatOpenAI

# 通过 OpenAI 兼容接口接入 Qwen
llm = ChatOpenAI(
    model="kimi-k2.5",
    openai_api_base=os.environ["Qwen_API_BASE"],
    openai_api_key=os.environ["Qwen_API_KEY"],
    temperature=0.7,
)

response = llm.invoke("用一句话解释什么是RL")
print(type(response))       # AIMessage 对象
print(response.content)     # 文本内容

### 刚才发生了什么？

1. `ChatOpenAI` 是 LangChain 对 OpenAI 兼容接口的封装。通义千问提供了这种兼容接口，所以只需指定 `openai_api_base` 和对应模型名即可接入。
2. `.invoke()` 是 LangChain 的统一调用方法（后面会详细讲）。传入字符串，返回一个 `AIMessage` 对象。
3. `AIMessage.content` 是模型返回的文本内容。

### 可用模型

| 模型名 | 说明 |
|--------|------|
| `qwen-turbo` | 速度最快，适合简单任务 |
| `qwen-plus` | 均衡选择，性价比高 |
| `qwen-max` | 能力最强，适合复杂推理 |

切换模型只需改 `model` 参数，其余代码不变——这正是统一接口的价值。

### 常见问题

- **`AuthenticationError`**：API Key 错误或未设置环境变量。检查 `echo $Qwen_API_KEY` 是否有输出。
- **`Connection Error`**：网络问题，确认能访问 `dashscope.aliyuncs.com`。
- **`model not found`**：模型名拼写错误，注意是 `qwen-plus` 不是 `qwen_plus`。

## 4. 对比：不用 LangChain vs 用 LangChain

为什么不直接用 `openai` SDK？先看「不用 LangChain」的写法：

In [ ]:
# ❌ 不用 LangChain：直接用 openai SDK
from openai import OpenAI

client = OpenAI(
    base_url=os.environ["Qwen_API_BASE"],
    api_key=os.environ["Qwen_API_KEY"],
)

resp = client.chat.completions.create(
    model="kimi-k2.5",
    messages=[{"role": "user", "content": "用一句话解释什么是大语言模型"}],
)

print(resp.choices[0].message.content)

In [16]:
# ✅ 用 LangChain：同样的事，但更简洁
response = llm.invoke("用一句话解释什么是大语言模型")
print(response.content)

大语言模型（LLM）是一种基于深度学习、通过在海量文本数据上进行训练而构建的巨型人工智能模型，能够理解、生成和推理自然语言，完成问答、写作、翻译、编程等多种语言任务。


### 对比总结

现在看起来只是少了几行代码，差别不大。但 LangChain 的真正价值在后续章节中体现：

- **Prompt Templates**：模板化管理提示词（第二章）
- **LCEL 链**：用 `|` 管道组合多个步骤（第三章）
- **Memory**：自动管理对话历史（第四章）
- **RAG / Agents**：文档检索、工具调用等复杂功能（第五六章）

这些都建立在统一的 Runnable 接口之上，直接用 openai SDK 则需要自己造轮子。

## 5. 消息类型（Message Types）

In [17]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# 构造多轮对话消息列表
messages = [
    SystemMessage(content="你是一位专业的 Python 编程老师，回答要简洁易懂。"),
    HumanMessage(content="列表推导式和 map 函数有什么区别？"),
]

response = llm.invoke(messages)
print(response.content)

列表推导式和 `map()` 的核心区别如下：

| 特性 | 列表推导式 | `map()` |
|------|-------------|----------|
| **返回值** | 直接返回**列表**（Python 3） | 返回**迭代器**（需 `list()` 转换才得列表） |
| **语法** | `[表达式 for x in 可迭代对象 if 条件]`（更直观、声明式） | `map(函数, 可迭代对象)`（函数式风格） |
| **过滤能力** | ✅ 支持 `if` 条件筛选（如 `[x*2 for x in lst if x>0]`） | ❌ 本身不支持过滤，需配合 `filter()` |
| **多参数/复杂逻辑** | ✅ 易写多层循环、嵌套逻辑 | ⚠️ 多参数需用 `lambda` 或 `itertools.starmap`，稍繁琐 |
| **可读性** | 简单转换/过滤时更清晰易懂 | 函数已存在时（如 `map(str, lst)`）简洁；复杂逻辑易变晦涩 |

✅ **推荐用列表推导式**：多数日常场景（尤其带条件或简单计算）——更 Pythonic、易读、高效。  
✅ **考虑用 `map()`**：当已有现成函数、且处理超大序列（节省内存，因返回迭代器）。

💡 小例子对比：
```python
nums = [1, 2, 3, 4]

# 列表推导式（推荐）
squares = [x**2 for x in nums if x % 2 == 0]  # [4, 16]

# map + filter（等价但啰嗦）
squares2 = list(map(lambda x: x**2, filter(lambda x: x % 2 == 0, nums)))  # [4, 16]
```

总结：**优先用列表推导式；追求内存效率或复用函数时再选 `map()`。**


### 刚才发生了什么？

LangChain 用结构化的消息对象代替纯字符串，每种消息类型对应 API 中的一个角色：

| 消息类型 | 角色 | 说明 |
|---------|------|------|
| `SystemMessage` | system | 系统提示，设定模型行为和身份 |
| `HumanMessage` | user | 用户输入 |
| `AIMessage` | assistant | 模型回复 |

前面直接传字符串 `llm.invoke("...")` 其实是快捷写法，LangChain 内部会自动包装成 `HumanMessage`。需要设定系统提示或构造多轮对话时，就得用消息列表。

In [18]:
# 查看 AIMessage 的完整结构
print("content:", response.content[:80], "...")
print("response_metadata:", list(response.response_metadata.keys()))
print("usage_metadata:", response.usage_metadata)  # Token 用量

content: 列表推导式和 `map()` 的核心区别如下：

| 特性 | 列表推导式 | `map()` |
|------|-------------|-------- ...
response_metadata: ['token_usage', 'model_provider', 'model_name', 'system_fingerprint', 'id', 'finish_reason', 'logprobs']
usage_metadata: {'input_tokens': 39, 'output_tokens': 456, 'total_tokens': 495, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}


`AIMessage` 不只有文本，还带有 token 用量、模型信息等元数据，方便做成本监控和调试。

## 6. 流式输出（Streaming）

In [19]:
# 流式调用：逐 token 输出
for chunk in llm.stream("用三句话介绍 Python 语言的优势"):
    print(chunk.content, end="", flush=True)

print()  # 换行

Python 语法简洁清晰，接近自然语言，大幅降低了学习和开发门槛，使开发者能更专注于解决问题而非语言细节。  
它拥有庞大且活跃的开源生态（如 NumPy、Pandas、TensorFlow、Django 等），覆盖数据科学、人工智能、Web 开发、自动化运维等几乎所有主流领域。  
Python 是跨平台、解释型语言，支持快速原型开发与迭代，并具备良好的可扩展性（可通过 C/C++ 扩展或与 Go/Java 等语言集成），兼顾开发效率与工程实用性。


### 刚才发生了什么？

`.stream()` 返回一个生成器，每次 yield 一个 `AIMessageChunk`。相比 `.invoke()` 等全部生成完再返回，流式输出的首 token 延迟更低，用户体验更好。

在后续章节中，用 LCEL 构建的链也能自动透传流式输出。

## 7. 批量调用（Batch）

In [20]:
# 批量调用：并行处理多个请求
questions = [
    "什么是 RAG？用一句话回答。",
    "什么是 Agent？用一句话回答。",
    "什么是向量数据库？用一句话回答。",
]

responses = llm.batch(questions)

for q, r in zip(questions, responses):
    print(f"Q: {q}")
    print(f"A: {r.content}")
    print("-" * 50)

Q: 什么是 RAG？用一句话回答。
A: RAG（Retrieval-Augmented Generation，检索增强生成）是一种将外部知识检索与大语言模型生成能力相结合的技术，通过先从可靠知识库中检索相关信息，再将其作为上下文输入模型来生成更准确、可溯源、实时性强的回答。
--------------------------------------------------
Q: 什么是 Agent？用一句话回答。
A: Agent（智能体）是一种能够感知环境、自主决策并采取行动以实现特定目标的软件实体。
--------------------------------------------------
Q: 什么是向量数据库？用一句话回答。
A: 向量数据库是一种专门用于高效存储、索引和检索高维向量（如由AI模型生成的嵌入向量）的数据库系统，支持基于相似度的近似最近邻（ANN）搜索。
--------------------------------------------------


### 刚才发生了什么？

`.batch()` 接收一个输入列表，**并行**发送请求，比逐个 `.invoke()` 快得多。

## 8. Runnable 统一接口

LangChain 中所有组件（模型、模板、解析器、链……）都实现了 **Runnable** 接口：

| 方法 | 说明 |
|------|------|
| `.invoke(input)` | 同步调用，返回单个结果 |
| `.stream(input)` | 流式调用，逐步返回 |
| `.batch(inputs)` | 批量调用，并行处理 |
| `.ainvoke(input)` | 异步调用 |
| `.astream(input)` | 异步流式 |

这意味着你学会了 `llm.invoke()` / `.stream()` / `.batch()`，后续章节中模板、链等组件的调用方式完全相同，不需要重新学习。

## 总结

本章学习了：
- ✅ 使用 uv 搭建 LangChain 开发环境
- ✅ 通过 OpenAI 兼容接口接入通义千问（Qwen）API
- ✅ 对比了直接用 openai SDK 与 LangChain 的区别
- ✅ `SystemMessage`、`HumanMessage`、`AIMessage` 三种消息类型
- ✅ `.invoke()`、`.stream()`、`.batch()` 三种调用方式
- ✅ Runnable 统一接口的概念

## 下一章

**第二章：Prompt Templates（提示词模板）** —— 学习如何管理和复用提示词，构建动态提示。